In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/Employee_Benefits_Guide_2026_v1.pdf")
docs = loader.load()

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(docs)

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

---

In [7]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(texts, embeddings)

In [8]:
query = "국가 기술 자격 중 기사 자격증을 취득하면 얼마를 받을 수 있을까?"

In [12]:
result = vectorstore.similarity_search(query, k=5)

In [13]:
len(result)

5

In [14]:
result

[Document(id='af6b1480-7122-477c-8624-6b34f1949600', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'page': 16, 'total_pages': 22, 'Author': '', 'CreationDate': "D:20260116002528+09'00'", 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft: Print To PDF', 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx'}, page_content='7.2 자격증 취득 지원\n직무 관련 국가 기술 자격 취득 시 축하금 및 수당을 지급합니다.\n자격 등급 축하금 (1 회성) 자격 수당 (월) 대상 자격증 예시\n기술사/기능장 200 만원 30 만원 금속재료, 용접, 기계가공 등\n기사 50 만원 10 만원 일반기계, 전기, 산업안전 등\n산업기사 30 만원 5 만원 기계설계, 위험물 등\n기능사 10 만원 3 만원 선반, 밀링, 특수용접 등\n\uf0b7 조건: 동일 등급 내 1 개 자격증만 수당 인정 (상위 등급 취득 시 갱신). 축하금은 횟수\n제한 없음.\n7.3 해외 연수 (Global Explorer)\n\uf0b7 대상: 연간 최우수 사원 (MVP) 및 우수 팀.\n\uf0b7 내용: 매년 10 월 독일/일본 등 선진 제조 현장 견학 및 문화 탐방 (7 박 9 일).\n\uf0b7 비용: 회사 전액 부담 (개인 경비 1,000 유로 별도 지급).\n8. 윤리 경영 및 보안 (Ethics & Security)\n8.1 직장 내 괴롭힘 방지 가이드'),
 Document(id='f889518a-5933-4c77-a7a7-904acb3b9264', met

---

로컬에 저장하기

In [15]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(texts, embeddings)

In [16]:
vectorstore.save_local("./faiss_index")

In [20]:
del vectorstore

In [ ]:
# 로컬에 저장된 데이터 로드
vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [23]:
vectorstore.similarity_search(query)

[Document(id='a037b5bd-5d83-40fd-87a0-f1333d783d8a', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'page': 16, 'total_pages': 22, 'Author': '', 'CreationDate': "D:20260116002528+09'00'", 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft: Print To PDF', 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx'}, page_content='7.2 자격증 취득 지원\n직무 관련 국가 기술 자격 취득 시 축하금 및 수당을 지급합니다.\n자격 등급 축하금 (1 회성) 자격 수당 (월) 대상 자격증 예시\n기술사/기능장 200 만원 30 만원 금속재료, 용접, 기계가공 등\n기사 50 만원 10 만원 일반기계, 전기, 산업안전 등\n산업기사 30 만원 5 만원 기계설계, 위험물 등\n기능사 10 만원 3 만원 선반, 밀링, 특수용접 등\n\uf0b7 조건: 동일 등급 내 1 개 자격증만 수당 인정 (상위 등급 취득 시 갱신). 축하금은 횟수\n제한 없음.\n7.3 해외 연수 (Global Explorer)\n\uf0b7 대상: 연간 최우수 사원 (MVP) 및 우수 팀.\n\uf0b7 내용: 매년 10 월 독일/일본 등 선진 제조 현장 견학 및 문화 탐방 (7 박 9 일).\n\uf0b7 비용: 회사 전액 부담 (개인 경비 1,000 유로 별도 지급).\n8. 윤리 경영 및 보안 (Ethics & Security)\n8.1 직장 내 괴롭힘 방지 가이드'),
 Document(id='58adccd3-80cb-4100-98a6-587683156ddb', met

---

In [ ]:
retriever = vectorstore.as_retriever() # runnable 객체

In [25]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000264EC437C80>, search_kwargs={})

In [30]:
result = retriever.invoke(query, k=3)

In [31]:
result

[Document(id='a037b5bd-5d83-40fd-87a0-f1333d783d8a', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'page': 16, 'total_pages': 22, 'Author': '', 'CreationDate': "D:20260116002528+09'00'", 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft: Print To PDF', 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx'}, page_content='7.2 자격증 취득 지원\n직무 관련 국가 기술 자격 취득 시 축하금 및 수당을 지급합니다.\n자격 등급 축하금 (1 회성) 자격 수당 (월) 대상 자격증 예시\n기술사/기능장 200 만원 30 만원 금속재료, 용접, 기계가공 등\n기사 50 만원 10 만원 일반기계, 전기, 산업안전 등\n산업기사 30 만원 5 만원 기계설계, 위험물 등\n기능사 10 만원 3 만원 선반, 밀링, 특수용접 등\n\uf0b7 조건: 동일 등급 내 1 개 자격증만 수당 인정 (상위 등급 취득 시 갱신). 축하금은 횟수\n제한 없음.\n7.3 해외 연수 (Global Explorer)\n\uf0b7 대상: 연간 최우수 사원 (MVP) 및 우수 팀.\n\uf0b7 내용: 매년 10 월 독일/일본 등 선진 제조 현장 견학 및 문화 탐방 (7 박 9 일).\n\uf0b7 비용: 회사 전액 부담 (개인 경비 1,000 유로 별도 지급).\n8. 윤리 경영 및 보안 (Ethics & Security)\n8.1 직장 내 괴롭힘 방지 가이드'),
 Document(id='58adccd3-80cb-4100-98a6-587683156ddb', met